In [1]:
!pip install pandas numpy scikit-learn

In [3]:
# ==========================================
# 1. IMPORT REQUIRED LIBRARIES
# ==========================================
# Standard libraries for data manipulation and statistical operations
import numpy as np
import pandas as pd

# Scikit-learn tools for model training, dataset splitting, and categorical encoding
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [4]:
# ==========================================
# 2. GENERATE SYNTHETIC HISTORICAL DATA
# ==========================================
# Set random seed for reproducible output across runs
np.random.seed(42)
num_samples = 1000

# Create historical customer interaction record fields
data = {
    'customer_id': np.random.randint(1001, 1050, size=num_samples),
    'product_id': np.random.randint(5001, 5020, size=num_samples),
    'product_category': np.random.choice(
        ['Electronics', 'Clothing', 'Home', 'Books'], size=num_samples
    ),
    'product_price': np.random.uniform(10, 500, size=num_samples).round(2),
    'views': np.random.randint(1, 20, size=num_samples),
    'cart_adds': np.random.randint(0, 10, size=num_samples),
    'purchased': np.random.choice([0, 1], size=num_samples, p=[0.7, 0.3]),
}

# Load dictionary into pandas Dataframe
df = pd.DataFrame(data)

# Introduce duplicate records and missing prices to demonstrate data cleaning
df = pd.concat([df, df.iloc[:5]], ignore_index=True)
df.loc[10:15, 'product_price'] = np.nan

# Display first 5 rows of raw dataset
df.head()

,customer_id,product_id,product_category,product_price,views,cart_adds,purchased
0,1039,5005,Clothing,131.02,5,8,0
1,1029,5004,Books,454.06,8,6,1
2,1015,5006,Electronics,132.28,7,1,0
3,1043,5002,Electronics,143.26,14,7,0
4,1008,5011,Clothing,382.11,3,6,0


In [5]:
# ==========================================
# 3. DATA CLEANING & FEATURE ENGINEERING
# ==========================================
# Remove exact duplicate rows from the dataset
df = df.drop_duplicates()

# Impute missing values in 'product_price' using median price
df['product_price'] = df['product_price'].fillna(df['product_price'].median())

# Feature Engineering: Calculate purchase intent score (cart adds relative to total views)
df['cart_to_view_ratio'] = df['cart_adds'] / (df['views'] + 1e-5)

# Encode text categories into numeric values for Machine Learning algorithms
le_category = LabelEncoder()
df['category_encoded'] = le_category.fit_transform(df['product_category'])

# Preview updated dataframe with processed features
df.head()

,customer_id,product_id,product_category,product_price,views,cart_adds,purchased,cart_to_view_ratio,category_encoded
0,1039,5005,Clothing,131.02,5,8,0,1.599997,1
1,1029,5004,Books,454.06,8,6,1,0.749999,0
2,1015,5006,Electronics,132.28,7,1,0,0.142857,2
3,1043,5002,Electronics,143.26,14,7,0,0.500000,2
4,1008,5011,Clothing,382.11,3,6,0,1.999993,1


In [10]:
# ==========================================
# 4. MODEL SELECTION & EVALUATION
# ==========================================
from sklearn.metrics import accuracy_score, classification_report

# Specify independent features (X) and target variable (y)
features = [
    'product_price',
    'views',
    'cart_adds',
    'cart_to_view_ratio',
    'category_encoded',
]
X = df[features]
y = df['purchased']

# Split data into 80% training set and 20% testing set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Initialize and train Random Forest Classifier model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on test data for evaluation
y_pred = model.predict(X_test)

# Display evaluation metrics
print('=== MODEL EVALUATION REPORT ===')
print(f'Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%\n')
print('Classification Report:')
print(classification_report(y_test, y_pred))

=== MODEL EVALUATION REPORT ===
Accuracy Score: 67.00%

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.83      0.79       146
           1       0.34      0.24      0.28        54

    accuracy                           0.67       200
   macro avg       0.54      0.53      0.53       200
weighted avg       0.64      0.67      0.65       200



In [14]:
# ==========================================
# 5. PERSONALIZED RECOMMENDATION SYSTEM ENGINE
# ==========================================
# Extract unique product information to build active product catalog
product_catalog = (
    df[['product_id', 'product_category', 'product_price']]
    .drop_duplicates()
    .reset_index(drop=True)
)


def recommend_top_5_personalized(customer_id):
  """Generates personalized product recommendations based on a customer's specific historical interactions."""
  # Get customer's historical activity from dataframe
  customer_history = df[df['customer_id'] == customer_id]

  # If customer exists in history, calculate their personalized average behavior
  if not customer_history.empty:
    avg_views = customer_history['views'].mean()
    avg_cart_adds = customer_history['cart_adds'].mean()
  else:
    # Fallback for brand new customers with no history
    avg_views = df['views'].median()
    avg_cart_adds = df['cart_adds'].median()

  candidate_products = product_catalog.copy()

  # Apply customer-specific behavior across candidate products
  candidate_products['views'] = avg_views
  candidate_products['cart_adds'] = avg_cart_adds
  candidate_products['cart_to_view_ratio'] = avg_cart_adds / (
      avg_views + 1e-5
  )
  candidate_products['category_encoded'] = le_category.transform(
      candidate_products['product_category']
  )

  # Predict purchase probability using customer profile + product features
  X_pred = candidate_products[features]
  candidate_products['purchase_probability'] = model.predict_proba(X_pred)[
      :, 1
  ]

  # Sort by predicted probability and return top 5
  top_5 = candidate_products.sort_values(
      by='purchase_probability', ascending=False
  ).head(5)

  return top_5[
      ['product_id', 'product_category', 'product_price', 'purchase_probability']
  ]

In [17]:
# ==========================================
# 6. EXECUTE RECOMMENDATION GENERATOR (USER INPUT)
# ==========================================
try:
  # Accept Customer ID directly from user keyboard input
  user_input = input('Enter Customer ID (e.g., 1005, 1012, 1040): ')
  customer_id_input = int(user_input)

  # Fetch personalized recommendations
  top_5_recommendations = recommend_top_5_personalized(
      customer_id=customer_id_input
  )

  # Display results formatted in Jupyter Notebook
  print(
      f'\nTop 5 Personalized Recommendations for Customer ID:'
      f' {customer_id_input}\n'
  )
  display(top_5_recommendations)

except ValueError:
  print('Invalid input! Please enter a numeric Customer ID.')

Enter Customer ID (e.g., 1005, 1012, 1040):  1008



Top 5 Personalized Recommendations for Customer ID: 1008



,product_id,product_category,product_price,purchase_probability
835,5005,Clothing,496.69,0.79
962,5004,Clothing,489.48,0.77
804,5017,Clothing,488.88,0.77
484,5017,Clothing,490.68,0.74
164,5004,Clothing,489.76,0.72


In [1]:
%%writefile app.py
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import streamlit as st

st.title("🛒 Smart Product Recommendation System")
st.write(
    "Enter a Customer ID to get personalized Top 5 product recommendations."
)


# Load & preprocess data
@st.cache_data
def load_and_train():
  np.random.seed(42)
  num_samples = 1000
  data = {
      "customer_id": np.random.randint(1001, 1050, size=num_samples),
      "product_id": np.random.randint(5001, 5020, size=num_samples),
      "product_category": np.random.choice(
          ["Electronics", "Clothing", "Home", "Books"], size=num_samples
      ),
      "product_price": np.random.uniform(10, 500, size=num_samples).round(2),
      "views": np.random.randint(1, 20, size=num_samples),
      "cart_adds": np.random.randint(0, 10, size=num_samples),
      "purchased": np.random.choice([0, 1], size=num_samples, p=[0.7, 0.3]),
  }
  df = pd.DataFrame(data)
  df = df.drop_duplicates()
  df["product_price"] = df["product_price"].fillna(df["product_price"].median())
  df["cart_to_view_ratio"] = df["cart_adds"] / (df["views"] + 1e-5)

  customer_prev = df.groupby("customer_id")["purchased"].sum().reset_index()
  customer_prev.columns = ["customer_id", "previous_purchases"]
  df = df.merge(customer_prev, on="customer_id", how="left")

  le_category = LabelEncoder()
  df["category_encoded"] = le_category.fit_transform(df["product_category"])

  features = [
      "product_price",
      "views",
      "cart_adds",
      "cart_to_view_ratio",
      "previous_purchases",
      "category_encoded",
  ]
  X, y = df[features], df["purchased"]

  model = RandomForestClassifier(n_estimators=100, random_state=42)
  model.fit(X, y)

  return df, model, le_category, features


df, model, le_category, features = load_and_train()

# UI Inputs
customer_id_input = st.number_input(
    "Enter Customer ID:", min_value=1001, max_value=1050, value=1005, step=1
)

if st.button("Generate Recommendations"):
  customer_history = df[df["customer_id"] == customer_id_input]

  if not customer_history.empty:
    avg_views = customer_history["views"].mean()
    avg_cart_adds = customer_history["cart_adds"].mean()
    prev_purchases = customer_history["previous_purchases"].iloc[0]
  else:
    avg_views = df["views"].median()
    avg_cart_adds = df["cart_adds"].median()
    prev_purchases = 0

  product_catalog = (
      df[["product_id", "product_category", "product_price"]]
      .drop_duplicates()
      .reset_index(drop=True)
  )
  candidate_products = product_catalog.copy()
  candidate_products["views"] = avg_views
  candidate_products["cart_adds"] = avg_cart_adds
  candidate_products["cart_to_view_ratio"] = avg_cart_adds / (avg_views + 1e-5)
  candidate_products["previous_purchases"] = prev_purchases
  candidate_products["category_encoded"] = le_category.transform(
      candidate_products["product_category"]
  )

  X_pred = candidate_products[features]
  candidate_products["purchase_probability"] = model.predict_proba(X_pred)[
      :, 1
  ]

  top_5 = candidate_products.sort_values(
      by="purchase_probability", ascending=False
  ).head(5)

  st.subheader(
      f"Top 5 Recommendations for Customer ID: {int(customer_id_input)}"
  )
  st.dataframe(
      top_5[[
          "product_id",
          "product_category",
          "product_price",
          "purchase_probability",
      ]]
  )

Writing app.py


In [2]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn

Writing requirements.txt
